# Test pysepal-api exports in the local SEPAL sandbox

Select **Python 3** and run the cells in order. This notebook submits two small exports (an image and a table) through SEPAL's task manager when you run the export cells. Results go into a unique directory under `~/module_results/ee_export_smoke_test/`.

The backend must include the `ee.export` adapter. The sandbox must have the local `~/1_modules/pysepal-api` checkout installed. Authentication uses the sandbox API key and its existing Earth Engine credentials; no credentials are stored in this notebook.

This local dev environment uses a certificate the sandbox does not trust. The setup cell explicitly disables TLS verification for this test notebook; use `verify=True` with a trusted deployment.


In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import ee
import pysepal_api
from pysepal_api import SepalClient

print("pysepal-api:", version("pysepal-api"), pysepal_api.__file__)
sepal = SepalClient(timeout=60, verify=False)
assert hasattr(sepal, "ee_exports"), "Install the local pysepal-api checkout, then restart the kernel."
run_name = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
output_dir = f"module_results/ee_export_smoke_test/{run_name}"
print("SEPAL:", sepal.base_url)
print("Workspace entries:", len(sepal.files.list(".")))
print("Output:", Path.home() / output_dir)


## Make tiny Earth Engine objects

`ee.Initialize()` uses the credentials and project already configured by SEPAL in this sandbox. No interactive authentication is needed. The test image contains a constant value of 7, over a small rectangle near Rome.


In [ ]:
ee.Initialize()
region = ee.Geometry.Rectangle([12.49, 41.89, 12.50, 41.90])
image = ee.Image.constant(7).rename("test_value").toInt16().clip(region)
plots = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([12.495, 41.895]), {"id": 1, "label": "export_smoke_test"})
])
print("Earth Engine objects ready")


## Export an image to the SEPAL workspace

Running the next cell submits an actual export. You can also follow it in SEPAL's **Tasks** panel.


In [ ]:
image_task = sepal.ee_exports.submit_image(
    image=image,
    region=region,
    destination="SEPAL",
    name=f"image_smoke_{run_name}",
    scale=100,
    crs="EPSG:4326",
    workspace_path=f"{output_dir}/image",
    filename_prefix="constant_seven",
)
print("Image task:", image_task.id, image_task.state.value)


In [ ]:
image_result = sepal.tasks.wait(image_task.id, poll_interval=3, timeout=600)
print("Image export:", image_result.state.value)
image_files = sorted((Path.home() / output_dir / "image").glob("*"))
for path in image_files:
    print(path.name, path.stat().st_size, "bytes")
assert any(path.suffix == ".tif" for path in image_files), "Expected at least one GeoTIFF"


## Export a table to the SEPAL workspace

This tests the same route with a FeatureCollection. CSV exports follow the existing SEPAL exporter behavior, which includes a `.geo` geometry column.


In [ ]:
table_task = sepal.ee_exports.submit_table(
    collection=plots,
    destination="SEPAL",
    name=f"table_smoke_{run_name}",
    workspace_path=f"{output_dir}/table",
    filename_prefix="test_plots",
    file_format="CSV",
    selectors=["id", "label"],
)
print("Table task:", table_task.id, table_task.state.value)


In [ ]:
table_result = sepal.tasks.wait(table_task.id, poll_interval=3, timeout=600)
print("Table export:", table_result.state.value)
table_files = sorted((Path.home() / output_dir / "table").glob("*.csv"))
assert table_files, "Expected a CSV file"
for path in table_files:
    print(path.name)
    print(path.read_text())


## Inspect or cancel a task

If a wait times out, the export may still be running. Inspect its details with the next cell, or uncomment the cancellation line for the task you want to stop. Stopping a task cancels its Earth Engine operation and cleans temporary storage after submission.

To try other destinations, use `destination="DRIVE"` with `drive_folder="pysepal_api_smoke_test"`, or `destination="GEE"` with your own `asset_id`. Those destinations require a linked Google account. Table `max_vertices` is currently supported only for Drive exports; ImageCollection assets use SEPAL's tiled recipe workflow.


In [ ]:
for task in (image_task, table_task):
    details = sepal.tasks.get(task.id, details=True)
    print(details.id, details.state.value, details.status_description)

# sepal.tasks.cancel(image_task.id)
# sepal.tasks.cancel(table_task.id)


In [ ]:
sepal.close()
print("Client closed. Exported files remain in", Path.home() / output_dir)
